# OpenPlaque — LAD origin backtracking

This fresh experiment avoids trying to solve the left coronary ostium first. It searches the **source CCTA** for a convincing proximal LAD segment using LAD-specific anatomy plus RCA-calibrated true orthogonal lumen QC, then backtracks toward the root and estimates the proximal LAD origin/takeoff.

The existing TotalSegmentator **aorta mask is used only as an anatomical constraint**. It is not a coronary segmentation. No LCX tracking and no plaque processing are performed here.


## Step 1 — Mount Google Drive


In [ ]:
# FIRST EXECUTABLE CELL
from google.colab import drive
drive.mount('/content/drive')


## Step 2 — Cache reuse controls


In [ ]:
REUSE_SOURCE_CT = True
REUSE_LAD_EVIDENCE = True
REUSE_RCA_CALIBRATION = True
REUSE_CANDIDATE_ROUTES = True
REUSE_CANDIDATE_QC = True
REUSE_FIGURES = True
REUSE_REPORT = True


## Step 3 — Install this branch and JPEG-lossless decoder


In [ ]:
!rm -rf /content/OpenPlaque
!git clone -q --branch lad-origin-backtrack-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%pip -q install pydicom SimpleITK scipy scikit-image matplotlib pandas psutil "pylibjpeg>=2.0" "pylibjpeg-libjpeg>=2.1"
import sys, os, gc, psutil
sys.path.insert(0, '/content/OpenPlaque/src')
def ram(label):
    p = psutil.Process(os.getpid())
    print(f'{label}: RSS {p.memory_info().rss/1024**3:.2f} GB')
ram('After install')


## Step 4 — Initialize workflow


In [ ]:
from openplaque.lad_origin_backtrack import LADOriginBacktrackWorkflow, ALGORITHM_VERSION
reuse = {
    'source_ct': REUSE_SOURCE_CT,
    'lad_evidence': REUSE_LAD_EVIDENCE,
    'rca_calibration': REUSE_RCA_CALIBRATION,
    'candidate_routes': REUSE_CANDIDATE_ROUTES,
    'candidate_qc': REUSE_CANDIDATE_QC,
    'figures': REUSE_FIGURES,
    'report': REUSE_REPORT,
}
wf = LADOriginBacktrackWorkflow('/content/drive/MyDrive/OpenPlaque', reuse=reuse)
print('Algorithm:', ALGORITHM_VERSION)
display(wf.cache_status())


## Step 5 — Load disk-backed source CCTA and build LAD evidence

The full source CCTA remains disk-backed. A roughly 1-mm local evidence volume is built around the aortic root/heart. The TotalSegmentator aorta mask is used only to exclude the aorta and measure distance from it.


In [ ]:
wf.load_source_ct()
wf.build_evidence()
gc.collect(); ram('After source CT + LAD evidence')
print('Evidence shape:', wf.evd['ct'].shape)
print('Evidence spacing z,y,x (mm):', wf.evd['spacing_zyx'])


## Step 6 — RCA calibration

The already accepted RCA source-volume centerline is the subject-specific positive calibration for coronary radius, centering, circularity, attenuation, and contrast. It is not manually labeled ground truth.


In [ ]:
rca = wf.calibrate_rca()
display(rca)
display(wf.rca_qc)


## Step 7 — Generate LAD-directed candidate routes

Distal candidates must lie inferior/anterior to the aortic-root reference, remain coronary-sized in the downsampled evidence, and connect to a proximal coronary-scale candidate without traversing the aorta or a large bright chamber. This is not a generic left-main search.


In [ ]:
routes = wf.build_candidate_routes()
display(routes.head(20))
gc.collect(); ram('After LAD candidate routes')


## Step 8 — True source-resolution LAD QC and origin estimate

Each leading route is tested in actual source-volume orthogonal planes. A path is not accepted merely because its graph score is high. The origin estimate is the earliest stable coronary-sized point on the root-oriented LAD path; it is an estimated proximal LAD takeoff, not a manually labeled bifurcation and not an aortic ostium.


In [ ]:
summary = wf.qc_candidates()
print('Best LAD summary:')
display(summary)
print('Estimated LAD origin/takeoff:')
display(wf.origin)
display(wf.best_qc)
gc.collect(); ram('After LAD QC')


## Step 9 — QC figures

The decisive figures are the top true orthogonal sections, the direct RCA-vs-LAD comparison, the LAD course in patient LPS coordinates, and the proximal 0–12 mm origin neighborhood.


In [ ]:
figs = wf.plot_qc()
for f in figs:
    print('Saved:', f)
gc.collect(); ram('After figures')


## Step 10 — Package report-back ZIP


In [ ]:
zip_path = wf.package()
print('Report ZIP:', zip_path)
print('Drive search: https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_LAD_ORIGIN_BACKTRACK_REPORT_BACK.zip')


## Step 11 — Report back

After the ZIP is written, return to ChatGPT and say **Retrieve and analyze**. Do not proceed to LCX or plaque processing unless the LAD segment and its proximal origin neighborhood are visually convincing.
